## **Stock Market Data Cleaning and Transformation Pipeline**

In [12]:
import yfinance as yf

df = yf.download("AAPL", start="2020-01-01", end="2024-01-01")

/tmp/ipykernel_4080/943275834.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("AAPL", start="2020-01-01", end="2024-01-01")
[*********************100%***********************]  1 of 1 completed


# **Project Overview**

This notebook implements a robust data pipeline for fetching, cleaning, transforming, and validating historical stock market data using the Yahoo Finance API (yfinance). The objective is to convert raw OHLCV (Open, High, Low, Close, Volume) data into a clean, consistent, and analysis-ready dataset suitable for exploratory data analysis, financial modeling, and machine learning applications.

# **Key Features**
Automated extraction of historical stock market data using the yfinance library.
Comprehensive data cleaning, including handling missing values and removing duplicate records.
Transformation of stock data into a structured, analysis-friendly format.
Feature engineering with derived attributes such as daily returns, price range, and date-based features.
Data type optimization to improve memory efficiency and computational performance.
Validation of data integrity and export of both raw and cleaned datasets for future analysis.
# **Project Details**
Owner: Anwesa Panja

Project: Stock Market Data Cleaning and Transformation Pipeline

# **1. Install & Import Libraries**
This section ensures all necessary Python libraries are installed and imported for the project. It conditionally installs yfinance if it's not already present in the environment.

In [13]:

import os
import pandas as pd
import numpy as np
import datetime as dt

# Install yfinance if not already installed
try:
    import yfinance as yf
    print("yfinance is already installed.")
except ImportError:
    print("yfinance not found, installing...")
    !pip install yfinance
    import yfinance as yf
    print("yfinance installed successfully.")

print("All required libraries imported successfully.")

yfinance is already installed.
All required libraries imported successfully.


# **2. Connect Google Drive**
This step connects Google Colab to Google Drive, allowing the notebook to access and store datasets, project files, and generated outputs.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
df = pd.read_csv("/content/drive/MyDrive/raw_stock_data.csv")

# **3. Project Configuration**
This step defines the project settings, including stock symbols, date range, file paths, and other configuration parameters required for data extraction and processing.

In [16]:
# List of stock tickers to fetch
STOCKS = ['AAPL', 'MSFT', 'SPY']

# Time range for data fetching (5 years ago from today)
END_DATE = dt.datetime.now()
START_DATE = END_DATE - dt.timedelta(days=5*365) # Approximately 5 years

# File paths for raw and cleaned data exports
RAW_DATA_PATH = 'raw_stock_data.csv'
CLEANED_DATA_PATH = 'cleaned_stock_data.csv'

print(f"Stocks to fetch: {STOCKS}")
print(f"Data start date: {START_DATE.strftime('%Y-%m-%d')}")
print(f"Data end date: {END_DATE.strftime('%Y-%m-%d')}")
print(f"Raw data will be saved to: {RAW_DATA_PATH}")
print(f"Cleaned data will be saved to: {CLEANED_DATA_PATH}")

Stocks to fetch: ['AAPL', 'MSFT', 'SPY']
Data start date: 2021-07-20
Data end date: 2026-07-19
Raw data will be saved to: raw_stock_data.csv
Cleaned data will be saved to: cleaned_stock_data.csv


## **4. Data Extraction from Yahoo Finance**
This step retrieves historical stock market data from Yahoo Finance using the yfinance library and stores it in a DataFrame for further cleaning and analysis.

In [17]:

try:
    # Fetch data, explicitly setting auto_adjust=False to get both 'Close' and 'Adj Close'
    raw_df = yf.download(STOCKS, start=START_DATE, end=END_DATE, auto_adjust=False)
    print(f"Successfully fetched data for {STOCKS} from {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}.")
except Exception as e:
    print(f"Error fetching data: {e}")
    raw_df = pd.DataFrame() # Create an empty DataFrame to avoid errors later

[*********************100%***********************]  3 of 3 completed

Successfully fetched data for ['AAPL', 'MSFT', 'SPY'] from 2021-07-20 to 2026-07-19.


# **5. Save Raw Stock Market Data**
This step saves the extracted raw stock market data to a CSV file, creating a backup of the original dataset for traceability and future reference.

In [18]:
if not raw_df.empty:
    try:
        raw_df.to_csv(RAW_DATA_PATH)
        print(f"Raw data saved to {RAW_DATA_PATH}")
    except Exception as e:
        print(f"Error saving raw data: {e}")
else:
    print("Raw DataFrame is empty, skipping saving.")

Raw data saved to raw_stock_data.csv


## **6. Preview Raw Dataset**
This step displays the first few rows of the raw dataset to verify that the stock market data has been successfully retrieved before preprocessing.



In [19]:
raw_df

Price        Adj Close                               Close              \
Ticker            AAPL        MSFT         SPY        AAPL        MSFT   
Date                                                                     
2021-07-20  142.460007  268.062500  402.881897  146.149994  279.320007   
2021-07-21  141.728897  270.058563  406.143799  145.399994  281.399994   
2021-07-22  143.093567  274.607697  406.994263  146.800003  286.140015   
2021-07-23  144.809143  277.995361  411.181488  148.559998  289.670013   
2021-07-26  145.228302  277.400269  412.190887  148.990005  289.049988   
...                ...         ...         ...         ...         ...   
2026-07-13  317.309998  390.989990  749.169983  317.309998  390.989990   
2026-07-14  314.859985  384.929993  751.830017  314.859985  384.929993   
2026-07-15  327.500000  395.630005  754.809998  327.500000  395.630005   
2026-07-16  333.260010  401.100006  750.719971  333.260010  401.100006   
2026-07-17  333.739990  393.820007  743.289978  333.739990  393.820007   

Price                         High                                 Low  \
Ticker             SPY        AAPL        MSFT         SPY        AAPL   
Date                                                                     
2021-07-20  431.059998  147.100006  280.970001  432.420013  142.960007   
2021-07-21  434.549988  146.130005  281.519989  434.700012  144.630005   
2021-07-22  435.459991  148.199997  286.420013  435.720001  145.809998   
2021-07-23  439.940002  148.720001  289.989990  440.299988  146.919998   
2021-07-26  441.019989  149.830002  289.690002  441.029999  147.699997   
...                ...         ...         ...         ...         ...   
2026-07-13  749.169983  323.450012  393.649994  753.909973  315.779999   
2026-07-14  751.830017  316.190002  388.190002  753.340027  311.910004   
2026-07-15  754.809998  328.730011  398.959991  755.580017  317.320007   
2026-07-16  750.719971  334.679993  405.989990  754.570007  326.790009   
2026-07-17  743.289978  334.989990  398.390015  747.289978  329.000000   

Price                                     Open                          \
Ticker            MSFT         SPY        AAPL        MSFT         SPY   
Date                                                                     
2021-07-20  276.260010  424.829987  143.460007  278.029999  425.679993   
2021-07-21  277.290009  431.010010  145.529999  278.899994  432.339996   
2021-07-22  283.420013  433.690002  145.940002  283.839996  434.739990   
2021-07-23  286.500000  436.790009  147.550003  287.369995  437.519989   
2021-07-26  286.640015  439.260010  148.270004  289.000000  439.309998   
...                ...         ...         ...         ...         ...   
2026-07-13  384.149994  748.000000  317.019989  387.750000  752.469971   
2026-07-14  378.649994  748.659973  313.760010  382.820007  750.909973   
2026-07-15  386.399994  750.200012  317.619995  387.799988  754.239990   
2026-07-16  392.049988  747.880005  328.010010  398.309998  752.760010   
2026-07-17  389.390015  740.799988  331.980011  394.859985  742.080017   

Price         Volume                      
Ticker          AAPL      MSFT       SPY  
Date                                      
2021-07-20  96350000  26259700  99608200  
2021-07-21  74993500  24364300  64724400  
2021-07-22  77338200  23384100  47878500  
2021-07-23  71447400  22768100  63766600  
2021-07-26  72434100  23176100  43719200  
...              ...       ...       ...  
2026-07-13  43257800  28914900  44013600  
2026-07-14  36336800  27863900  35143100  
2026-07-15  60957600  36253700  43844800  
2026-07-16  62970600  37047400  46409800  
2026-07-17  63365300  33010300  62569200  

[1254 rows x 18 columns]

# **7. Check Raw Data Shape**
This step displays the number of rows and columns in the raw dataset, providing a quick overview of its size before preprocessing.

In [20]:
raw_df.shape

(1254, 18)

# **8. Raw_Dataset Overview**
This step provides an overview of the raw dataset by displaying its contents, allowing an initial inspection of the downloaded stock market data before any preprocessing.

In [21]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1254 entries, 2021-07-20 to 2026-07-17
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   (Adj Close, AAPL)  1254 non-null   float64
 1   (Adj Close, MSFT)  1254 non-null   float64
 2   (Adj Close, SPY)   1254 non-null   float64
 3   (Close, AAPL)      1254 non-null   float64
 4   (Close, MSFT)      1254 non-null   float64
 5   (Close, SPY)       1254 non-null   float64
 6   (High, AAPL)       1254 non-null   float64
 7   (High, MSFT)       1254 non-null   float64
 8   (High, SPY)        1254 non-null   float64
 9   (Low, AAPL)        1254 non-null   float64
 10  (Low, MSFT)        1254 non-null   float64
 11  (Low, SPY)         1254 non-null   float64
 12  (Open, AAPL)       1254 non-null   float64
 13  (Open, MSFT)       1254 non-null   float64
 14  (Open, SPY)        1254 non-null   float64
 15  (Volume, AAPL)     1254 non-null   int64  
 16  (Volum

# **9. Statistical Summary of the Dataset**
This step generates descriptive statistics for the dataset, providing a summary of key numerical measures such as count, mean, standard deviation, minimum, maximum, and quartiles.

In [22]:
raw_df.describe()

Price     Adj Close                                  Close               \
Ticker         AAPL         MSFT          SPY         AAPL         MSFT   
count   1254.000000  1254.000000  1254.000000  1254.000000  1254.000000   
mean     195.402958   359.872430   502.783475   197.362209   366.284258   
std       46.035045    83.259918   113.915967    45.135551    81.340373   
min      122.933548   207.733994   339.378510   125.019997   214.250000   
25%      158.059494   284.342224   407.773666   161.110004   295.010002   
50%      183.969063   368.570862   462.662979   186.099998   373.184998   
75%      227.325882   418.293015   589.042542   229.000000   423.364998   
max      333.739990   538.658569   757.618225   333.739990   542.070007   

Price                       High                                    Low  \
Ticker          SPY         AAPL         MSFT          SPY         AAPL   
count   1254.000000  1254.000000  1254.000000  1254.000000  1254.000000   
mean     517.329004   199.329968   369.726396   520.078988   195.209641   
std      107.489104    45.496033    81.629654   107.538538    44.731063   
min      356.559998   127.769997   220.410004   359.820007   124.169998   
25%      429.602509   162.967495   297.487503   432.835007   159.109997   
50%      477.514999   187.470001   376.125000   478.904999   184.065002   
75%      598.812515   230.842506   426.842506   601.242493   226.892502   
max      759.570007   334.989990   555.450012   760.400024   329.000000   

Price                                    Open                            \
Ticker         MSFT          SPY         AAPL         MSFT          SPY   
count   1254.000000  1254.000000  1254.000000  1254.000000  1254.000000   
mean     362.584976   514.105837   197.148852   366.283094   517.209824   
std       81.055379   107.298061    45.074012    81.459511   107.493734   
min      213.429993   348.109985   126.010002   217.550003   349.209991   
25%      292.730011   426.172485   161.037495   295.344994   429.907509   
50%      369.425003   475.990005   185.974998   373.514999   477.779999   
75%      418.970009   596.484985   229.145000   423.827507   598.769989   
max      540.770020   756.750000   331.980011   555.229980   758.150024   

Price         Volume                              
Ticker          AAPL          MSFT           SPY  
count   1.254000e+03  1.254000e+03  1.254000e+03  
mean    6.486522e+07  2.661515e+07  7.547158e+07  
std     2.855357e+07  1.207494e+07  2.912729e+07  
min     1.791060e+07  5.855900e+06  2.604870e+07  
25%     4.568418e+07  1.909282e+07  5.625435e+07  
50%     5.678100e+07  2.375105e+07  7.099910e+07  
75%     7.665268e+07  3.078940e+07  8.932838e+07  
max     3.186799e+08  1.862016e+08  2.566114e+08

# **10. Check Total Missing Values**
This step calculates the total number of missing values in the dataset, providing an overall assessment of data completeness before cleaning.

In [23]:

raw_df.isnull().sum().sum()

np.int64(0)

# **11. Check Data Types**
This step displays the data type of each column to verify that the dataset is correctly formatted before data cleaning and transformation.

In [47]:
raw_df.dtypes

Price      Ticker
Adj Close  AAPL      float64
           MSFT      float64
           SPY       float64
Close      AAPL      float64
           MSFT      float64
           SPY       float64
High       AAPL      float64
           MSFT      float64
           SPY       float64
Low        AAPL      float64
           MSFT      float64
           SPY       float64
Open       AAPL      float64
           MSFT      float64
           SPY       float64
Volume     AAPL        int64
           MSFT        int64
           SPY         int64
dtype: object

# **12. Check Missing Values in Each Column**
This step displays the number of missing values in each column, helping identify which features require data cleaning or imputation.

In [25]:
raw_df.isna().sum()

Price      Ticker
Adj Close  AAPL      0
           MSFT      0
           SPY       0
Close      AAPL      0
           MSFT      0
           SPY       0
High       AAPL      0
           MSFT      0
           SPY       0
Low        AAPL      0
           MSFT      0
           SPY       0
Open       AAPL      0
           MSFT      0
           SPY       0
Volume     AAPL      0
           MSFT      0
           SPY       0
dtype: int64

# **13. Preview Raw Dataset after Missing**
This step displays the first few rows of the dataset after handling missing values to verify that the data has been cleaned correctly.

In [26]:
raw_df

Price        Adj Close                               Close              \
Ticker            AAPL        MSFT         SPY        AAPL        MSFT   
Date                                                                     
2021-07-20  142.460007  268.062500  402.881897  146.149994  279.320007   
2021-07-21  141.728897  270.058563  406.143799  145.399994  281.399994   
2021-07-22  143.093567  274.607697  406.994263  146.800003  286.140015   
2021-07-23  144.809143  277.995361  411.181488  148.559998  289.670013   
2021-07-26  145.228302  277.400269  412.190887  148.990005  289.049988   
...                ...         ...         ...         ...         ...   
2026-07-13  317.309998  390.989990  749.169983  317.309998  390.989990   
2026-07-14  314.859985  384.929993  751.830017  314.859985  384.929993   
2026-07-15  327.500000  395.630005  754.809998  327.500000  395.630005   
2026-07-16  333.260010  401.100006  750.719971  333.260010  401.100006   
2026-07-17  333.739990  393.820007  743.289978  333.739990  393.820007   

Price                         High                                 Low  \
Ticker             SPY        AAPL        MSFT         SPY        AAPL   
Date                                                                     
2021-07-20  431.059998  147.100006  280.970001  432.420013  142.960007   
2021-07-21  434.549988  146.130005  281.519989  434.700012  144.630005   
2021-07-22  435.459991  148.199997  286.420013  435.720001  145.809998   
2021-07-23  439.940002  148.720001  289.989990  440.299988  146.919998   
2021-07-26  441.019989  149.830002  289.690002  441.029999  147.699997   
...                ...         ...         ...         ...         ...   
2026-07-13  749.169983  323.450012  393.649994  753.909973  315.779999   
2026-07-14  751.830017  316.190002  388.190002  753.340027  311.910004   
2026-07-15  754.809998  328.730011  398.959991  755.580017  317.320007   
2026-07-16  750.719971  334.679993  405.989990  754.570007  326.790009   
2026-07-17  743.289978  334.989990  398.390015  747.289978  329.000000   

Price                                     Open                          \
Ticker            MSFT         SPY        AAPL        MSFT         SPY   
Date                                                                     
2021-07-20  276.260010  424.829987  143.460007  278.029999  425.679993   
2021-07-21  277.290009  431.010010  145.529999  278.899994  432.339996   
2021-07-22  283.420013  433.690002  145.940002  283.839996  434.739990   
2021-07-23  286.500000  436.790009  147.550003  287.369995  437.519989   
2021-07-26  286.640015  439.260010  148.270004  289.000000  439.309998   
...                ...         ...         ...         ...         ...   
2026-07-13  384.149994  748.000000  317.019989  387.750000  752.469971   
2026-07-14  378.649994  748.659973  313.760010  382.820007  750.909973   
2026-07-15  386.399994  750.200012  317.619995  387.799988  754.239990   
2026-07-16  392.049988  747.880005  328.010010  398.309998  752.760010   
2026-07-17  389.390015  740.799988  331.980011  394.859985  742.080017   

Price         Volume                      
Ticker          AAPL      MSFT       SPY  
Date                                      
2021-07-20  96350000  26259700  99608200  
2021-07-21  74993500  24364300  64724400  
2021-07-22  77338200  23384100  47878500  
2021-07-23  71447400  22768100  63766600  
2021-07-26  72434100  23176100  43719200  
...              ...       ...       ...  
2026-07-13  43257800  28914900  44013600  
2026-07-14  36336800  27863900  35143100  
2026-07-15  60957600  36253700  43844800  
2026-07-16  62970600  37047400  46409800  
2026-07-17  63365300  33010300  62569200  

[1254 rows x 18 columns]

# **14. Verify Missing Values in the Dataset**
This section checks whether the dataset contains any missing (NaN) values, returning:

True → At least one missing value exists.

False → No missing values are present.


In [27]:
raw_df.isna().values.any()

np.False_

# **15. Handle Missing Values and Inspect Column Names**
This section:

Imputes missing values using the forward-fill (ffill) method.

Displays the column names to verify the dataset schema after cleaning.

In [28]:
raw_df = raw_df.ffill()

In [29]:

raw_df.columns

MultiIndex([('Adj Close', 'AAPL'),
            ('Adj Close', 'MSFT'),
            ('Adj Close',  'SPY'),
            (    'Close', 'AAPL'),
            (    'Close', 'MSFT'),
            (    'Close',  'SPY'),
            (     'High', 'AAPL'),
            (     'High', 'MSFT'),
            (     'High',  'SPY'),
            (      'Low', 'AAPL'),
            (      'Low', 'MSFT'),
            (      'Low',  'SPY'),
            (     'Open', 'AAPL'),
            (     'Open', 'MSFT'),
            (     'Open',  'SPY'),
            (   'Volume', 'AAPL'),
            (   'Volume', 'MSFT'),
            (   'Volume',  'SPY')],
           names=['Price', 'Ticker'])

# **16. Verify Dataset Structure After Data Cleaning**
This step verifies the dataset structure after data cleaning by checking the column names, data types, and non-null values to ensure the data is ready for further processing.

In [30]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1254 entries, 2021-07-20 to 2026-07-17
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   (Adj Close, AAPL)  1254 non-null   float64
 1   (Adj Close, MSFT)  1254 non-null   float64
 2   (Adj Close, SPY)   1254 non-null   float64
 3   (Close, AAPL)      1254 non-null   float64
 4   (Close, MSFT)      1254 non-null   float64
 5   (Close, SPY)       1254 non-null   float64
 6   (High, AAPL)       1254 non-null   float64
 7   (High, MSFT)       1254 non-null   float64
 8   (High, SPY)        1254 non-null   float64
 9   (Low, AAPL)        1254 non-null   float64
 10  (Low, MSFT)        1254 non-null   float64
 11  (Low, SPY)         1254 non-null   float64
 12  (Open, AAPL)       1254 non-null   float64
 13  (Open, MSFT)       1254 non-null   float64
 14  (Open, SPY)        1254 non-null   float64
 15  (Volume, AAPL)     1254 non-null   int64  
 16  (Volum

# **17. Restructure and Flatten the Dataset**
Steps Performed:----

1. Create a Copy of the Cleaned Dataset:-----

Create a duplicate of the cleaned DataFrame to preserve the original dataset.

2. Reset the Index:---

Convert the Date index into a regular column for easier manipulation and analysis.

3. Flatten MultiIndex Columns:---

Merge multi-level column names into a single level by joining the column names with an underscore (_), creating a simpler column structure.

4. Rename the Date Column:-----

Rename the Date_ column to Date for better readability and consistency.

5. Verify the Transformed Dataset:----

Display the first few rows of the DataFrame using head() to confirm that the restructuring has been performed successfully.

In [31]:
df_cleaned = raw_df.copy()
# Reset index to make 'Date' a regular column
df_cleaned = df_cleaned.reset_index()

In [32]:
# Flatten the MultiIndex columns
df_cleaned.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in df_cleaned.columns.values]

# Rename the 'Date' column to 'Date' (it might be 'Date_' after joining if it was a MultiIndex level)
df_cleaned = df_cleaned.rename(columns={'Date_': 'Date'})

In [33]:
# Display the first few rows of the cleaned DataFrame to verify
display(df_cleaned.head())

,Date,Adj Close_AAPL,Adj Close_MSFT,Adj Close_SPY,Close_AAPL,Close_MSFT,Close_SPY,High_AAPL,High_MSFT,High_SPY,Low_AAPL,Low_MSFT,Low_SPY,Open_AAPL,Open_MSFT,Open_SPY,Volume_AAPL,Volume_MSFT,Volume_SPY
0,2021-07-20,142.460007,268.062500,402.881897,146.149994,279.320007,431.059998,147.100006,280.970001,432.420013,142.960007,276.260010,424.829987,143.460007,278.029999,425.679993,96350000,26259700,99608200
1,2021-07-21,141.728897,270.058563,406.143799,145.399994,281.399994,434.549988,146.130005,281.519989,434.700012,144.630005,277.290009,431.010010,145.529999,278.899994,432.339996,74993500,24364300,64724400
2,2021-07-22,143.093567,274.607697,406.994263,146.800003,286.140015,435.459991,148.199997,286.420013,435.720001,145.809998,283.420013,433.690002,145.940002,283.839996,434.739990,77338200,23384100,47878500
3,2021-07-23,144.809143,277.995361,411.181488,148.559998,289.670013,439.940002,148.720001,289.989990,440.299988,146.919998,286.500000,436.790009,147.550003,287.369995,437.519989,71447400,22768100,63766600
4,2021-07-26,145.228302,277.400269,412.190887,148.990005,289.049988,441.019989,149.830002,289.690002,441.029999,147.699997,286.640015,439.260010,148.270004,289.000000,439.309998,72434100,23176100,43719200


In [34]:
# Display the new columns
display(df_cleaned.columns)

Index(['Date', 'Adj Close_AAPL', 'Adj Close_MSFT', 'Adj Close_SPY',
       'Close_AAPL', 'Close_MSFT', 'Close_SPY', 'High_AAPL', 'High_MSFT',
       'High_SPY', 'Low_AAPL', 'Low_MSFT', 'Low_SPY', 'Open_AAPL', 'Open_MSFT',
       'Open_SPY', 'Volume_AAPL', 'Volume_MSFT', 'Volume_SPY'],
      dtype='object')

In [35]:
df_cleaned.shape

(1254, 19)

#18. For reducing the Memory usage  
**Steps Performed**:---
1. Create a copy of the cleaned dataset.
2. Transform the dataset from wide format to long format using melt().
3. Extract Stock Name and Metric from the column names.
4. Remove the temporary variable column.
5. Reshape the data using pivot_table() to create separate metric columns.
6. Flatten the column hierarchy.
7. Rename and reorder the columns.
8. Update the cleaned dataset with the restructured data.
9. Convert numeric columns to appropriate data types.
10. Display the transformed dataset for verification.



In [36]:
import pandas as pd

# Make a copy to ensure we're working with a fresh state if previous ops modified it
df_to_transform = df_cleaned.copy()

# 1. Melt the DataFrame from wide to long format
# Identify columns to melt: all columns except 'Date'
columns_to_melt = [col for col in df_to_transform.columns if col != 'Date']
melted_df = df_to_transform.melt(id_vars=['Date'], value_vars=columns_to_melt, var_name='variable', value_name='value')

print("Intermediate: Melted DataFrame head:")
display(melted_df.head())

# 2. Extract 'Stock_name' and 'Metric' from the 'variable' column
def split_variable_column(col_name):
    # Splits on the last underscore, e.g., 'Adj Close_AAPL' -> ['Adj Close', 'AAPL']
    parts = col_name.rsplit('_', 1)
    if len(parts) == 2:
        metric = parts[0]
        stock_name = parts[1]
        return pd.Series([metric, stock_name])
    return pd.Series([col_name, '']) # Fallback, though should not be hit for data columns

melted_df[['Metric', 'Stock_name']] = melted_df['variable'].apply(split_variable_column)

# Drop the intermediary 'variable' column as it's no longer needed
melted_df = melted_df.drop(columns=['variable'])

print("Intermediate: Melted DataFrame after splitting variable and stock names:")
display(melted_df.head())

# 3. Pivot the DataFrame to get metrics as new columns
df_restructured = melted_df.pivot_table(
    index=['Date', 'Stock_name'],
    columns='Metric',
    values='value'
).reset_index()

# Flatten the columns after pivot, removing the 'Metric' name from the columns MultiIndex
df_restructured.columns.name = None

print("Intermediate: Pivoted DataFrame head:")
display(df_restructured.head())

# 4. Clean up column names and order
# Rename 'Adj Close' to 'Adj_Close' to match the user's request
df_restructured = df_restructured.rename(columns={'Adj Close': 'Adj_Close'})

# Define the desired order of columns as per user's request
desired_columns_order = ['Date', 'Open', 'Stock_name', 'High', 'Low', 'Close', 'Adj_Close', 'Volume']

# Filter and reorder columns, only keeping those that exist in the DataFrame
df_restructured = df_restructured[[col for col in desired_columns_order if col in df_restructured.columns]]

# 5. Overwrite df_cleaned with the newly restructured DataFrame
df_cleaned = df_restructured

# 6. Apply type conversions, adapted for the new structure
# Convert float64 columns to float32
for col in df_cleaned.select_dtypes("float64").columns:
    df_cleaned[col] = df_cleaned[col].astype("float32")

# Convert the 'Volume' column to int32 if it exists and is a float type.
# Volume was originally int64, then became float due to melting, so convert back to int32.
if 'Volume' in df_cleaned.columns and pd.api.types.is_float_dtype(df_cleaned['Volume']):
    # Ensure there are no NaNs in 'Volume' before converting to integer
    df_cleaned['Volume'] = df_cleaned['Volume'].astype("int32")


Intermediate: Melted DataFrame head:


,Date,variable,value
0,2021-07-20,Adj Close_AAPL,142.460007
1,2021-07-21,Adj Close_AAPL,141.728897
2,2021-07-22,Adj Close_AAPL,143.093567
3,2021-07-23,Adj Close_AAPL,144.809143
4,2021-07-26,Adj Close_AAPL,145.228302


Intermediate: Melted DataFrame after splitting variable and stock names:


,Date,value,Metric,Stock_name
0,2021-07-20,142.460007,Adj Close,AAPL
1,2021-07-21,141.728897,Adj Close,AAPL
2,2021-07-22,143.093567,Adj Close,AAPL
3,2021-07-23,144.809143,Adj Close,AAPL
4,2021-07-26,145.228302,Adj Close,AAPL


Intermediate: Pivoted DataFrame head:


,Date,Stock_name,Adj Close,Close,High,Low,Open,Volume
0,2021-07-20,AAPL,142.460007,146.149994,147.100006,142.960007,143.460007,96350000.0
1,2021-07-20,MSFT,268.062500,279.320007,280.970001,276.260010,278.029999,26259700.0
2,2021-07-20,SPY,402.881897,431.059998,432.420013,424.829987,425.679993,99608200.0
3,2021-07-21,AAPL,141.728897,145.399994,146.130005,144.630005,145.529999,74993500.0
4,2021-07-21,MSFT,270.058563,281.399994,281.519989,277.290009,278.899994,24364300.0


# **19. Verify Column Data Types**
This step verifies that each column has the correct data type, ensuring the dataset is properly formatted for analysis and further processing.


In [37]:
df_cleaned.dtypes

,0
Date,datetime64[ns]
Open,float32
Stock_name,object
High,float32
Low,float32
Close,float32
Adj_Close,float32
Volume,int32


# **20. Inspect Cleaned Dataset Structure**
This step displays the dataset's structure, including column names, data types, non-null values, and memory usage, to verify the integrity of the cleaned data.

In [38]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3762 entries, 0 to 3761
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        3762 non-null   datetime64[ns]
 1   Open        3762 non-null   float32       
 2   Stock_name  3762 non-null   object        
 3   High        3762 non-null   float32       
 4   Low         3762 non-null   float32       
 5   Close       3762 non-null   float32       
 6   Adj_Close   3762 non-null   float32       
 7   Volume      3762 non-null   int32         
dtypes: datetime64[ns](1), float32(5), int32(1), object(1)
memory usage: 147.1+ KB


# **21. Preview the Cleaned Dataset**
This step displays the first few rows of the cleaned dataset to verify that the data has been transformed and structured correctly before further analysis.

In [39]:
display(df_cleaned.head())

,Date,Open,Stock_name,High,Low,Close,Adj_Close,Volume
0,2021-07-20,143.460007,AAPL,147.100006,142.960007,146.149994,142.460007,96350000
1,2021-07-20,278.029999,MSFT,280.970001,276.260010,279.320007,268.062500,26259700
2,2021-07-20,425.679993,SPY,432.420013,424.829987,431.059998,402.881897,99608200
3,2021-07-21,145.529999,AAPL,146.130005,144.630005,145.399994,141.728897,74993504
4,2021-07-21,278.899994,MSFT,281.519989,277.290009,281.399994,270.058563,24364300


# **22. Check Cleaned Dataset Dimensions**
This step displays the number of rows and columns in the cleaned dataset to verify its dimensions after the transformation process.

In [40]:
df_cleaned.shape


(3762, 8)

# **23. Remove Duplicate Records**
This step identifies and removes duplicate rows from the dataset, ensuring that each record is unique and improving the overall data quality.

In [41]:
# Remove duplicate rows
initial_rows = df_cleaned.shape[0]
df_cleaned = df_cleaned.drop_duplicates()
final_rows = df_cleaned.shape[0]

print(f"Number of rows before removing duplicates: {initial_rows}")
print(f"Number of rows after removing duplicates: {final_rows}")
print(f"Number of duplicate rows removed: {initial_rows - final_rows}")

Number of rows before removing duplicates: 3762
Number of rows after removing duplicates: 3762
Number of duplicate rows removed: 0


# **24. Verify Dataset Dimensions After Deduplication**
This step checks the number of rows and columns in the dataset after removing duplicate records to verify the updated dataset size.

In [42]:
print("Shape of DataFrame after removing duplicates:")
display(df_cleaned.shape)

Shape of DataFrame after removing duplicates:


(3762, 8)

# **25. Preview Dataset After Deduplication**
This step displays the first five rows of the dataset after removing duplicate records to verify that the data has been cleaned correctly.

In [43]:
print("First 5 rows of DataFrame after removing duplicates:")
display(df_cleaned.head())

First 5 rows of DataFrame after removing duplicates:


,Date,Open,Stock_name,High,Low,Close,Adj_Close,Volume
0,2021-07-20,143.460007,AAPL,147.100006,142.960007,146.149994,142.460007,96350000
1,2021-07-20,278.029999,MSFT,280.970001,276.260010,279.320007,268.062500,26259700
2,2021-07-20,425.679993,SPY,432.420013,424.829987,431.059998,402.881897,99608200
3,2021-07-21,145.529999,AAPL,146.130005,144.630005,145.399994,141.728897,74993504
4,2021-07-21,278.899994,MSFT,281.519989,277.290009,281.399994,270.058563,24364300


# **26. Generate Statistical Summary of the Cleaned Dataset**
This step generates descriptive statistics for the cleaned dataset, providing an overview of key numerical measures such as count, mean, standard deviation, minimum, maximum, and quartiles to summarize the data.

In [44]:
if not df_cleaned.empty:
    print("Final Cleaned Data Description:")
    display(df_cleaned.describe())
else:
    print("DataFrame is empty, skipping final describe display.")

Final Cleaned Data Description:


,Date,Open,High,Low,Close,Adj_Close,Volume
count,3762,3762.000000,3762.000000,3762.000000,3762.000000,3762.000000,3.762000e+03
mean,2024-01-15 11:26:41.913875712,360.213928,363.045135,357.300140,360.325195,352.686279,5.565065e+07
min,2021-07-20 00:00:00,126.010002,127.769997,124.169998,125.019997,122.933548,5.855900e+06
25%,2022-10-14 00:00:00,228.022495,230.409996,225.787498,228.022503,225.921394,3.002968e+07
50%,2024-01-16 12:00:00,370.599991,373.745010,366.565002,370.610001,359.031509,5.003445e+07
75%,2025-04-16 00:00:00,451.205002,453.685005,449.252502,451.627510,434.819023,7.362363e+07
max,2026-07-17 00:00:00,758.150024,760.400024,756.750000,759.570007,757.618225,3.186799e+08
std,NaN,154.380371,154.712265,153.813583,154.329620,152.039536,3.229931e+07


# **27. Validate Final Dataset Structure**
This step verifies that the cleaned dataset has the correct column names and structure, ensuring it matches the required format before further analysis.

In [45]:

if not df_cleaned.empty:
    expected_cols = ['Date', 'Stock_Name', 'Open', 'High', 'Low', 'Close', 'Adj_Close', 'Volume']
    if set(df_cleaned.columns) == set(expected_cols) and len(df_cleaned.columns) == len(expected_cols):
        print("Status: Data structure matches the required long format.")
    else:
        print("Status: Data structure does NOT match the required long format. Check column names and count.")
        print("Expected columns:", sorted(expected_cols))
        print("Actual columns:", sorted(df_cleaned.columns.tolist()))
    print("Final validation completed.")
else:
    print("DataFrame is empty, skipping final data structure validation.")

Status: Data structure does NOT match the required long format. Check column names and count.
Expected columns: ['Adj_Close', 'Close', 'Date', 'High', 'Low', 'Open', 'Stock_Name', 'Volume']
Actual columns: ['Adj_Close', 'Close', 'Date', 'High', 'Low', 'Open', 'Stock_name', 'Volume']
Final validation completed.


# **28. Export and Download the Cleaned Dataset**
Workflow:---
1. Import the Google Colab file handling module.
2. Convert the cleaned DataFrame into a CSV file.
3. Save the file in the Colab environment.
4. Download the CSV file to your local system.

In [46]:
from google.colab import files

# Save the cleaned DataFrame to a CSV file
df_cleaned.to_csv('cleaned_stock_data.csv', index=False)

# Offer the file for download
files.download('cleaned_stock_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>